Copyright 2021 Google LLC. SPDX-License-Identifier: Apache-2.0

Licensed under the Apache License, Version 2.0 (the "License"); you may not use this file except in compliance with the License. You may obtain a copy of the License at

https://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software distributed under the License is distributed on an "AS IS" BASIS, WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied. See the License for the specific language governing permissions and limitations under the License.

# Code as Policies：卓上マニピュレーション インタラクティブデモ（Gemini API 版・日本語解説付き）

このNotebookは、論文 [Code as Policies: Language Model Programs for Embodied Control](https://code-as-policies.github.io/)（Liang et al., 2022）の公式デモ（論文 IV.D 節のシミュレーション卓上タスク）をもとにしています。

**元のNotebookからの主な変更点**

| 項目 | 元のNotebook | このNotebook |
|---|---|---|
| 言語モデル | OpenAI `text-davinci-003`（提供終了） | **Google Gemini API**（`google-genai` SDK） |
| API呼び出し | `openai.Completion.create` | `client.models.generate_content` |
| バグ修正機能 | `openai.Edit`（`code-davinci-edit-001`） | Gemini に修正を指示するプロンプト |
| 動画表示 | `moviepy` | `imageio` で mp4 を作り HTML で表示 |
| 解説 | 英語・最小限 | 各ステップに日本語の解説セル |

## Code as Policies とは？

ロボットへの自然言語の指示（例：「赤いブロックを一番右のボウルの左に置いて」）を、**LLM に Python コードとして書かせ、そのコードを実行してロボットを動かす**手法です。

- 人間が書いた「コメント（指示）→ コード」の例（few-shot プロンプト）を LLM に見せると、新しい指示に対するコードを書いてくれる
- 生成されたコードは、`put_first_on_second()`（物体を掴んで別の場所に置く）のような **ロボット用 API（関数）** を呼び出す
- コードなので `for` 文・`if` 文・`numpy` 計算などを使って、空間的な推論や繰り返し動作を表現できる
- LLM が「まだ定義されていない関数」を使った場合、**その関数自体も LLM に書かせる**（階層的コード生成）

## このNotebookの流れ

1. Gemini API キーの設定・ライブラリのインストール
2. Gemini をコード補完エンジンとして使うための関数を作る
3. LMP（Language Model Program）の中核クラスを実装する
4. PyBullet によるロボットシミュレーション環境を用意する
5. LMP に見せるプロンプト（few-shot 例）と設定を用意する
6. すべてを組み合わせて、自然言語でロボットを動かす

**使い方：** 上から順にセルを実行し、最後の「インタラクティブデモ」で指示を入力してください（Google Colab での実行を想定しています）。

## 0. 全体のアーキテクチャ

このデモでは、役割の異なる複数の LMP（Language Model Program = LLM + プロンプト + 実行環境）が連携します。

```
ユーザーの指示 "put the red block left of the rightmost bowl"
        │
        ▼
┌──────────────────────┐
│  tabletop_ui (LMP)   │  高レベルの指示 → ロボット操作コードを生成
└──────────────────────┘
        │ 生成されたコードの中で呼び出される
        ├──► parse_obj_name      : "一番右のボウル" → 'blue bowl' など物体名を特定
        ├──► parse_position      : "〜の左10cm" → 座標 [x, y] を計算
        ├──► parse_question      : "〜は何個？" などの質問に答える
        └──► transform_shape_pts : 点群の拡大・回転・平行移動
        │
        ▼ 生成コード中に未定義の関数があれば…
┌──────────────────────┐
│     fgen (LMPFGen)   │  関数シグネチャから関数本体を生成（再帰的に）
└──────────────────────┘
        │
        ▼
 put_first_on_second() などのロボットAPI → PyBullet シミュレータでロボットが動く
```

ポイントは **「LMP が別の LMP を関数として呼び出せる」** ことです。`tabletop_ui` が生成したコードの中で `parse_position('a point 10cm left of the green bowl')` と書くと、その文字列が `parse_position` LMP への新しい指示になり、座標を返すコードが生成・実行されます。

## 1. Gemini API キーの設定

1. [Google AI Studio](https://aistudio.google.com/apikey) で API キーを発行します。
2. 次のいずれかの方法でキーを設定します。
   - **（推奨）Colab のシークレット機能**：左サイドバーの🔑アイコンから `GEMINI_API_KEY` という名前で登録し、「ノートブックからのアクセス」をオンにする
   - 環境変数 `GEMINI_API_KEY` に設定する
   - 下のセルの `'YOUR KEY HERE'` を直接書き換える（Notebookを共有する場合はキーが漏れるので注意）

`MODEL_NAME` で使うモデルを指定します。コード生成は速度と精度のバランスが良い Flash 系で十分動きます。最新のモデル名は [Gemini API のモデル一覧](https://ai.google.dev/gemini-api/docs/models) で確認してください。

In [ ]:
import os

# --- APIキーの読み込み（Colabシークレット → 環境変数 → 直書き の順に探す）---
gemini_api_key = 'YOUR KEY HERE'
try:
    from google.colab import userdata
    gemini_api_key = userdata.get('GEMINI_API_KEY')
except Exception:
    gemini_api_key = os.environ.get('GEMINI_API_KEY', gemini_api_key)

# --- 使用するGeminiモデル ---
MODEL_NAME = 'gemini-3.5-flash-lite'

## 2. ライブラリのインストールとインポート

| ライブラリ | 用途 |
|---|---|
| `google-genai` | Gemini API の公式 Python SDK |
| `pybullet` | 物理シミュレータ（UR5e ロボットアーム＋グリッパー） |
| `shapely` | 線・多角形などの 2D 幾何計算（「三角形に並べる」などで LLM が使う） |
| `astunparse` | Python の構文木（AST）をソースコードに戻す（未定義関数の検出に使う） |
| `pygments` | 生成されたコードを色付きで表示 |
| `imageio`, `imageio-ffmpeg` | シミュレーションの様子を mp4 動画にする |
| `gdown` | Google Drive からロボットのモデルファイルをダウンロード |

In [ ]:
!pip install -q google-genai numpy scipy shapely astunparse pygments pybullet imageio imageio-ffmpeg opencv-python gdown

必要なモジュールをインポートします。`from shapely.geometry import *` のようにワイルドカードでインポートしているのは、後で LLM が生成するコードの中から `LineString` や `rotate` などを自由に使えるようにするためです。

Colab 以外（ローカルの Jupyter など）でも動くように、画像表示関数 `cv2_imshow` は matplotlib による代替を用意しています。

In [ ]:
import re
import ast
import copy
import base64
import threading
from time import sleep

import numpy as np
import cv2
import imageio
import pybullet
import pybullet_data
from IPython.display import HTML, display

# LMP（生成コード）から使う幾何ライブラリ
import shapely
import astunparse
from shapely.geometry import *
from shapely.affinity import *

# 生成コードを色付き表示するため
from pygments import highlight
from pygments.lexers import PythonLexer
from pygments.formatters import TerminalFormatter

# Gemini API
from google import genai
from google.genai import types
from google.genai import errors as genai_errors

# 画像表示（Colab なら専用関数、それ以外は matplotlib で代用）
try:
    from google.colab.patches import cv2_imshow
except ImportError:
    import matplotlib.pyplot as plt
    def cv2_imshow(img_bgr):
        plt.imshow(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB))
        plt.axis('off')
        plt.show()

## 3. Gemini を「コード補完エンジン」として使う

### なぜ工夫が必要か

元の実装で使っていた `text-davinci-003` は **補完（Completion）型** のモデルでした。プロンプト（few-shot 例＋最後の指示コメント）を渡すと、その「続き」のコードだけを出力し、`stop` に指定した文字列（例：`#`）が現れたところで止まります。

一方 Gemini は **チャット型** のモデルなので、そのまま使うと次のような問題が起こりがちです。

- 出力が ```` ```python ```` のようなマークダウンのコードブロックで囲まれる
- 最後の指示コメント（`# place the red block ...`）を繰り返してから書き始める
- 「以下がコードです」のような説明文が付く
- `import numpy as np` などを書き足してしまう（このデモでは安全のため `import` を含むコードは実行を拒否します）

### 対策

1. **システム指示（system instruction）** で「あなたは補完エンジンです。続きのコードだけを出力してください」と伝える
2. **後処理** で、コードブロック記号の除去・指示の繰り返しの除去・`import` 行の除去を行う
3. **stop トークンの再現**：API 側の `stop_sequences` を使うと「最初に指示コメントを繰り返した瞬間（`#`）に止まって空になる」ことがあるため、出力を受け取ってから **行頭が stop トークンで始まる行で切り捨てる** 方式にしています
4. `temperature=0` で出力をなるべく決定的にする
5. **思考（thinking）を最小限にする**：`gemini-3.5-flash-lite` などの 3 系モデルは思考がオンになっていて完全にはオフにできないので、`thinking_level='minimal'` で最小にします（2.5 系なら `thinking_budget=0` でオフ）。思考トークンも `max_output_tokens` に含まれ、足りないとコードが途中で切れたり空になったりするので、上限に `THINKING_TOKEN_HEADROOM` の余裕を足しています

レート制限（HTTP 429）やサーバーエラー（5xx）が返ってきた場合は、10 秒待って再試行します。

In [ ]:
client = genai.Client(api_key=gemini_api_key)

# Gemini に「チャット」ではなく「コード補完」として振る舞ってもらうためのシステム指示
CODE_COMPLETION_SYSTEM_INSTRUCTION = """You are a Python code completion engine.
The user message is a Python script made of example pairs: a comment describing a task, followed by the code that does it.
Continue the script by writing ONLY the Python code for the LAST comment.
Rules:
- Output raw Python code only. No markdown, no code fences, no explanations.
- Do not repeat the last comment line.
- Do not write any import statements; everything needed is already imported.
- Use only the functions and variables that appear in the examples.
- Stop after the code for the last comment. Do not write new comments or new tasks."""

RETRYABLE_STATUS_CODES = (429, 500, 502, 503, 504)


def _extract_code(text):
    """```python ... ``` で囲まれていれば中身だけを取り出す。"""
    m = re.search(r'```(?:python|py)?[ \t]*\n(.*?)(?:```|$)', text, re.S)
    return m.group(1) if m else text


def _postprocess(text, prompt, stop_tokens):
    """チャット型モデルの出力を、補完型モデルの出力に近い形へ整える。"""
    text = _extract_code(text)
    last_prompt_line = prompt.strip().split('\n')[-1].strip()

    out_lines = []
    for line in text.split('\n'):
        stripped = line.strip()
        # 先頭の空行や、最後の指示コメントの繰り返しは読み飛ばす
        if not out_lines and (stripped == '' or stripped == last_prompt_line):
            continue
        # stop トークンで始まる行が来たら、そこで出力を打ち切る（補完APIの stop を再現）
        if any(stripped.startswith(tok) for tok in stop_tokens):
            break
        # import 文は exec_safe で拒否されるので取り除く
        if stripped.startswith('import ') or (stripped.startswith('from ') and ' import ' in stripped):
            continue
        out_lines.append(line)
    return '\n'.join(out_lines).strip()


# 思考（thinking）トークンも max_output_tokens に含まれるので、その分の余裕を足しておく
THINKING_TOKEN_HEADROOM = 1024


def _thinking_config(model):
    """思考をできるだけ減らす設定（速く・安く・出力が途中で切れないように）。"""
    if model.startswith('gemini-2.5'):
        # 2.5 系は thinking_budget で指定（Flash / Flash-Lite は 0 でオフ、Pro は最小 128）
        return types.ThinkingConfig(thinking_budget=0 if 'flash' in model else 128)
    # 3 系以降は thinking_level で指定（オフにはできないので最小の minimal にする）
    return types.ThinkingConfig(thinking_level='minimal')


def gemini_complete(prompt, stop=(), temperature=0.0, max_tokens=512, model=None,
                    system_instruction=CODE_COMPLETION_SYSTEM_INSTRUCTION):
    """Gemini でプロンプトの「続きのコード」を生成して返す。"""
    model = model or MODEL_NAME
    config = types.GenerateContentConfig(
        system_instruction=system_instruction,
        temperature=temperature,
        max_output_tokens=max_tokens + THINKING_TOKEN_HEADROOM,
        thinking_config=_thinking_config(model),
    )

    while True:
        try:
            response = client.models.generate_content(model=model, contents=prompt, config=config)
            break
        except genai_errors.APIError as e:
            if e.code in RETRYABLE_STATUS_CODES:
                print(f'Gemini API got err {e}')
                print('Retrying after 10s.')
                sleep(10)
            else:
                raise

    return _postprocess(response.text or '', prompt, list(stop))

### 動作確認

簡単な few-shot プロンプトで、Gemini が「続きのコードだけ」を返すか確かめてみます。`# define function` の行で止まり、`def` から始まるコードだけが表示されれば OK です。

In [ ]:
test_prompt = """
# define function: total = get_total(xs=numbers).
def get_total(xs):
    return np.sum(xs)

# define function: y = eval_line(x, slope, y_intercept=0).
""".strip()

print(gemini_complete(test_prompt, stop=['# define', '# example']))

## 4. ロボットのモデルファイル（アセット）をダウンロード

PyBullet でロボットを表示・シミュレーションするために、形状や関節を記述した **URDF ファイル** が必要です。

- `ur5e/` … Universal Robots 社の 6 軸アームロボット UR5e
- `robotiq_2f_85/` … Robotiq 社の 2 本指グリッパー
- `bowl/` … ボウル（ブロックは PyBullet の箱形状で作るのでファイル不要）

すでにダウンロード済みならスキップします。

In [ ]:
if not os.path.exists('ur5e/ur5e.urdf'):
    !gdown 1Cc_fDSBL6QiDvNT4dpfAEbhbALSVoWcc
    !gdown 1yOMEm-Zp_DL3nItG9RozPeJAmeOldekX
    !gdown 1GsqNLhEl9dd4Mc3BM0dX3MibOI1FVWNM
    !unzip -q ur5e.zip
    !unzip -q robotiq_2f_85.zip
    !unzip -q bowl.zip

## 5. LMP（Language Model Program）の中核を実装する

ここが Code as Policies の心臓部です。次の 4 つを順番に作ります。

| 部品 | 役割 |
|---|---|
| `merge_dicts` / `var_exists` / `exec_safe` | 補助関数（辞書の結合・変数の存在確認・生成コードの安全な実行） |
| `FunctionParser` | 生成コードの構文木を調べ、**呼び出されている関数の一覧** を取り出す |
| `LMP` | プロンプトを組み立てて Gemini にコードを生成させ、実行する |
| `LMPFGen` | 未定義の関数を見つけたら、その **関数本体を Gemini に生成** させる |

### 5.1 補助関数

`exec_safe` は LLM が生成したコードを `exec()` で実行する関数です。LLM の出力は信頼できないので、最低限の安全策として次の対策をしています。

- `import` や `__`（`__import__` や `__class__` などの特殊属性）を含むコードは実行しない
- 生成コードの中から `exec` / `eval` を呼べないよう、何もしない関数に差し替える

※ これは完全なサンドボックスではありません。研究デモ用の簡易的な対策です。

`exec(code, gvars, lvars)` の `gvars` は「グローバル変数（生成コードから使える関数や変数）」、`lvars` は「ローカル変数（生成コードが作った変数が入る）」の辞書です。この仕組みで **生成コードに使わせる API を辞書で制御** しています。

In [ ]:
def merge_dicts(dicts):
    """複数の辞書を1つにまとめる（後ろの辞書が優先）。"""
    return {
        k: v
        for d in dicts
        for k, v in d.items()
    }


def var_exists(name, all_vars):
    """name という名前の変数・関数が all_vars の中に存在するか。"""
    try:
        eval(name, all_vars)
    except:
        exists = False
    else:
        exists = True
    return exists


def exec_safe(code_str, gvars=None, lvars=None):
    """LLMが生成したコードを、最低限の安全チェックをしてから実行する。"""
    banned_phrases = ['import', '__']
    for phrase in banned_phrases:
        assert phrase not in code_str, f'生成コードに禁止語 "{phrase}" が含まれています:\n{code_str}'

    if gvars is None:
        gvars = {}
    if lvars is None:
        lvars = {}
    # 生成コードの中から exec / eval を使えないようにする
    empty_fn = lambda *args, **kwargs: None
    custom_gvars = merge_dicts([
        gvars,
        {'exec': empty_fn, 'eval': empty_fn}
    ])
    exec(code_str, custom_gvars, lvars)

### 5.2 FunctionParser：生成コードから「呼ばれている関数」を見つける

Python 標準の `ast`（抽象構文木）モジュールを使って、コードを文字列のまま解析します。

例えば LLM が次のコードを生成したとします。

```python
block_positions = get_obj_positions_np(block_names)
closest_idx = get_closest_idx(points=block_positions, point=get_obj_pos('purple bowl'))
```

`FunctionParser` はこれを走査して、

- `visit_Call`：関数呼び出しを見つけるたびに `{関数名: 呼び出し部分の文字列}` を記録
  - 例：`'get_closest_idx' → "get_closest_idx(points=block_positions, point=get_obj_pos('purple bowl'))"`
- `visit_Assign`：`x = f(...)` という代入文なら `{関数名: 代入文全体}` を記録
  - 例：`'get_obj_positions_np' → "block_positions = get_obj_positions_np(block_names)"`

代入文の形のほうが「戻り値が何を表すか（`block_positions`）」まで分かるので、関数を生成するときのヒントとして優先して使います。

In [ ]:
class FunctionParser(ast.NodeTransformer):

    def __init__(self, fs, f_assigns):
        super().__init__()
        self._fs = fs                # 関数名 -> 呼び出し式 の文字列
        self._f_assigns = f_assigns  # 関数名 -> 代入文 の文字列

    def visit_Call(self, node):
        self.generic_visit(node)  # 入れ子になった呼び出しも先に処理する
        if isinstance(node.func, ast.Name):  # obj.method() ではなく func() の形だけを対象にする
            f_sig = astunparse.unparse(node).strip()
            f_name = astunparse.unparse(node.func).strip()
            self._fs[f_name] = f_sig
        return node

    def visit_Assign(self, node):
        self.generic_visit(node)
        if isinstance(node.value, ast.Call):  # 右辺が関数呼び出しの代入文
            assign_str = astunparse.unparse(node).strip()
            f_name = astunparse.unparse(node.value.func).strip()
            self._f_assigns[f_name] = assign_str
        return node

### 5.3 LMP クラス：プロンプト → コード生成 → 実行

`LMP` は「1 つの役割を持った言語モデルプログラム」です。呼び出されると次の処理を行います。

1. **`build_prompt`**：プロンプトを組み立てる
   - few-shot 例（ベースプロンプト）
   - `maintain_session=True` なら、これまでに実行したコードの履歴（`exec_hist`）→ 「さっきの操作を元に戻して」のような文脈依存の指示に対応できる
   - `context`（例：`objects = ['red block', 'blue bowl', ...]`）
   - 指示を `# ` と `.` で挟んでコメントにしたもの（例：`# put the red block on the blue bowl.`）
2. **Gemini でコードを生成**（前の章で作った `gemini_complete`）
3. **未定義関数の生成**：`LMPFGen.create_new_fs_from_code` で、生成コードが使っているのにまだ存在しない関数を作る
4. **実行**：`exec_safe` で生成コードを実行する。`fixed_vars`（numpy や shapely）と `variable_vars`（ロボット API・他の LMP・生成済み関数）を合わせてグローバル変数として渡す
5. **戻り値**：`has_return=True` の LMP（`parse_position` など）は、生成コードが変数 `ret_val` に入れた値を返す

`variable_vars` の中に **他の LMP 自身** も入れておくことで、生成コードから `parse_obj_name('the blocks', ...)` のように LMP を関数として呼び出せる、というのが巧妙な点です。

In [ ]:
class LMP:

    def __init__(self, name, cfg, lmp_fgen, fixed_vars, variable_vars):
        self._name = name
        self._cfg = cfg

        self._base_prompt = self._cfg['prompt_text']  # few-shot 例
        self._stop_tokens = list(self._cfg['stop'])

        self._lmp_fgen = lmp_fgen            # 未定義関数を生成する LMPFGen
        self._fixed_vars = fixed_vars        # 変わらない変数（np, shapely など）
        self._variable_vars = variable_vars  # 変わる変数（ロボットAPI, 他のLMP, 生成した関数）
        self.exec_hist = ''                  # これまでに実行したコードの履歴

    def clear_exec_hist(self):
        self.exec_hist = ''

    def build_prompt(self, query, context=''):
        # プロンプト中の {variable_vars_imports} を、使える関数名の import 文に置き換える
        if len(self._variable_vars) > 0:
            variable_vars_imports_str = f"from utils import {', '.join(self._variable_vars.keys())}"
        else:
            variable_vars_imports_str = ''
        prompt = self._base_prompt.replace('{variable_vars_imports}', variable_vars_imports_str)

        # 会話の文脈を保つ LMP なら、過去の実行履歴を付け足す
        if self._cfg['maintain_session']:
            prompt += f'\n{self.exec_hist}'

        # 物体リストなどの文脈
        if context != '':
            prompt += f'\n{context}'

        # 指示をコメントの形にして末尾に置く（例: "# put the red block on the blue bowl."）
        use_query = f'{self._cfg["query_prefix"]}{query}{self._cfg["query_suffix"]}'
        prompt += f'\n{use_query}'

        return prompt, use_query

    def __call__(self, query, context='', **kwargs):
        prompt, use_query = self.build_prompt(query, context=context)

        # ① Gemini にコードを生成させる
        code_str = gemini_complete(
            prompt,
            stop=self._stop_tokens,
            temperature=self._cfg['temperature'],
            max_tokens=self._cfg['max_tokens'],
            model=self._cfg['model'],
        )

        # 実行するコード（context も一緒に実行して objects 変数などを定義する）とログ用の文字列
        if self._cfg['include_context'] and context != '':
            to_exec = f'{context}\n{code_str}'
            to_log = f'{context}\n{use_query}\n{code_str}'
        else:
            to_exec = code_str
            to_log = f'{use_query}\n{to_exec}'

        to_log_pretty = highlight(to_log, PythonLexer(), TerminalFormatter())
        print(f'LMP {self._name} exec:\n\n{to_log_pretty}\n')

        # ② 生成コードが使っている未定義の関数を、LMPFGen に作らせて登録する
        new_fs = self._lmp_fgen.create_new_fs_from_code(code_str)
        self._variable_vars.update(new_fs)

        # ③ 生成コードを実行する
        gvars = merge_dicts([self._fixed_vars, self._variable_vars])
        lvars = kwargs  # 呼び出し時のキーワード引数（例: shape_pts=...）はローカル変数として渡る

        if not self._cfg['debug_mode']:
            exec_safe(to_exec, gvars, lvars)

        self.exec_hist += f'\n{to_exec}'

        # 会話を保つ LMP は、生成コードが作った変数を次回以降も使えるようにする
        if self._cfg['maintain_session']:
            self._variable_vars.update(lvars)

        # ④ 戻り値を持つ LMP は ret_val などの変数の値を返す
        if self._cfg['has_return']:
            return lvars[self._cfg['return_val_name']]

### 5.4 LMPFGen：未定義の関数を LLM に書かせる（階層的コード生成）

few-shot 例の中では、`get_obj_positions_np` や `get_closest_idx`、`make_triangle` など **どこにも定義されていない関数** が使われています。これらは実行時に `LMPFGen` が生成します。

**`create_new_fs_from_code(code_str)` の流れ**

1. `FunctionParser` で `code_str` 中の関数呼び出しをすべて集める
2. 既に存在する関数（`var_exists`）はスキップ
3. 存在しない関数について、`create_f_from_sig` で関数本体を生成
   - プロンプト末尾は `# define function: closest_idx = get_closest_idx(points=..., point=...).` のような形になる
   - Gemini が `def get_closest_idx(points, point): ...` を書く
4. 生成した関数の **本体の中にも未定義の関数があれば、再帰的に生成** する
5. 子関数が作られたら、親関数を再定義して子関数を参照できるようにする

`fix_bugs=True` の場合は、生成した関数をもう一度 Gemini に渡して「バグがあれば直して」と頼みます（元の実装では OpenAI の Edit API を使っていた部分です。デモではデフォルトで無効です）。

In [ ]:
FIX_BUGS_SYSTEM_INSTRUCTION = """You fix bugs in Python functions.
Fix the bug if there is one. Improve readability. Keep same inputs and outputs. Only small changes. No comments.
Output only the corrected Python code. No markdown, no code fences, no explanations."""


class LMPFGen:

    def __init__(self, cfg, fixed_vars, variable_vars):
        self._cfg = cfg

        self._stop_tokens = list(self._cfg['stop'])
        self._fixed_vars = fixed_vars
        self._variable_vars = variable_vars

        self._base_prompt = self._cfg['prompt_text']

    def create_f_from_sig(self, f_name, f_sig, other_vars=None, fix_bugs=False, return_src=False):
        """関数シグネチャ（例: 'y = f(x)'）から関数本体を生成し、関数オブジェクトを返す。"""
        print(f'Creating function: {f_sig}')

        use_query = f'{self._cfg["query_prefix"]}{f_sig}{self._cfg["query_suffix"]}'
        prompt = f'{self._base_prompt}\n{use_query}'

        f_src = gemini_complete(
            prompt,
            stop=self._stop_tokens,
            temperature=self._cfg['temperature'],
            max_tokens=self._cfg['max_tokens'],
            model=self._cfg['model'],
        )

        if fix_bugs:
            # 生成した関数を Gemini にレビュー・修正させる
            f_src = gemini_complete(
                f_src,
                temperature=0,
                max_tokens=self._cfg['max_tokens'],
                model=self._cfg['model'],
                system_instruction=FIX_BUGS_SYSTEM_INSTRUCTION,
            )

        if other_vars is None:
            other_vars = {}
        gvars = merge_dicts([self._fixed_vars, self._variable_vars, other_vars])
        lvars = {}

        # def 文を実行すると、lvars に関数オブジェクトが入る
        exec_safe(f_src, gvars, lvars)

        f = lvars[f_name]

        to_print = highlight(f'{use_query}\n{f_src}', PythonLexer(), TerminalFormatter())
        print(f'LMP FGEN created:\n\n{to_print}\n')

        if return_src:
            return f, f_src
        return f

    def create_new_fs_from_code(self, code_str, other_vars=None, fix_bugs=False, return_src=False):
        """code_str の中で使われている未定義関数をすべて生成して辞書で返す。"""
        # ① 呼び出されている関数を集める（代入文の形があればそちらを優先）
        fs, f_assigns = {}, {}
        f_parser = FunctionParser(fs, f_assigns)
        f_parser.visit(ast.parse(code_str))
        for f_name, f_assign in f_assigns.items():
            if f_name in fs:
                fs[f_name] = f_assign

        if other_vars is None:
            other_vars = {}

        new_fs = {}
        srcs = {}
        for f_name, f_sig in fs.items():
            all_vars = merge_dicts([self._fixed_vars, self._variable_vars, new_fs, other_vars])
            # ② まだ存在しない関数だけを生成する
            if not var_exists(f_name, all_vars):
                f, f_src = self.create_f_from_sig(f_name, f_sig, new_fs, fix_bugs=fix_bugs, return_src=True)

                # ③ 生成した関数の本体にも未定義の関数があれば、再帰的に生成する
                f_def_body = astunparse.unparse(ast.parse(f_src).body[0].body)
                child_fs, child_f_srcs = self.create_new_fs_from_code(
                    f_def_body, other_vars=all_vars, fix_bugs=fix_bugs, return_src=True
                )

                if len(child_fs) > 0:
                    new_fs.update(child_fs)
                    srcs.update(child_f_srcs)

                    # ④ 子関数が見えるスコープで親関数を定義し直す
                    gvars = merge_dicts([self._fixed_vars, self._variable_vars, new_fs, other_vars])
                    lvars = {}

                    exec_safe(f_src, gvars, lvars)

                    f = lvars[f_name]

                new_fs[f_name], srcs[f_name] = f, f_src

        if return_src:
            return new_fs, srcs
        return new_fs

## 6. 卓上シミュレーション環境（PyBullet）

ここからは LLM とは独立した、ロボットのシミュレーション環境の部分です。UR5e アームと 2 本指グリッパーで、机の上のブロックやボウルを掴んで動かします。

### 6.1 定数の定義

- `COLORS`：物体の色（RGBA）。10 色のブロックとボウルを用意
- `CORNER_POS`：作業台の角・辺・中央の座標
- `ALL_BLOCKS` / `ALL_BOWLS`：シーンに置ける物体の名前の一覧
- `BOUNDS`：作業領域の範囲（X: -0.3〜0.3 m, Y: -0.8〜-0.2 m, Z: 0〜0.15 m）。ロボットの土台が原点

座標系は **ロボットから見て右が +X、奥（上）が +Y** です。プロンプト中の「top」は Y が大きい側、「left」は X が小さい側を指します。

In [ ]:
# # Global constants: pick and place objects, colors, workspace bounds
COLORS = {
    'blue':   (78/255,  121/255, 167/255, 255/255),
    'red':    (255/255,  87/255,  89/255, 255/255),
    'green':  (89/255,  169/255,  79/255, 255/255),
    'orange': (242/255, 142/255,  43/255, 255/255),
    'yellow': (237/255, 201/255,  72/255, 255/255),
    'purple': (176/255, 122/255, 161/255, 255/255),
    'pink':   (255/255, 157/255, 167/255, 255/255),
    'cyan':   (118/255, 183/255, 178/255, 255/255),
    'brown':  (156/255, 117/255,  95/255, 255/255),
    'gray':   (186/255, 176/255, 172/255, 255/255),
}

CORNER_POS = {
  'top left corner':     (-0.3 + 0.05, -0.2 - 0.05, 0),
  'top side':            (0,           -0.2 - 0.05, 0),
  'top right corner':    (0.3 - 0.05,  -0.2 - 0.05, 0),
  'left side':           (-0.3 + 0.05, -0.5,        0),
  'middle':              (0,           -0.5,        0),
  'right side':          (0.3 - 0.05,  -0.5,        0),
  'bottom left corner':  (-0.3 + 0.05, -0.8 + 0.05, 0),
  'bottom side':         (0,           -0.8 + 0.05, 0),
  'bottom right corner': (0.3 - 0.05,  -0.8 + 0.05, 0),
}

ALL_BLOCKS = ['blue block', 'red block', 'green block', 'orange block', 'yellow block', 'purple block', 'pink block', 'cyan block', 'brown block', 'gray block']
ALL_BOWLS = ['blue bowl', 'red bowl', 'green bowl', 'orange bowl', 'yellow bowl', 'purple bowl', 'pink bowl', 'cyan bowl', 'brown bowl', 'gray bowl']

PIXEL_SIZE = 0.00267857
BOUNDS = np.float32([[-0.3, 0.3], [-0.8, -0.2], [0, 0.15]])  # X Y Z

### 6.2 グリッパー（Robotiq 2F-85）

`Robotiq2F85` クラスはグリッパーの開閉と接触判定を担当します。

- `activate()` / `release()`：指を閉じる（掴む）/ 開く（離す）
- `step()`：別スレッドで動き続け、複数ある指関節が連動して動くよう拘束をかける
- `detect_contact()`：物を置くとき、下ろしている物体が何かに接触したかを判定（接触したら離す）
- `check_grasp()` / `grasp_width()`：指の間隔から、物を掴めているかを判定

In [ ]:
# Gripper (Robotiq 2F85) code

class Robotiq2F85:
  """Gripper handling for Robotiq 2F85."""

  def __init__(self, robot, tool):
    self.robot = robot
    self.tool = tool
    pos = [0.1339999999999999, -0.49199999999872496, 0.5]
    rot = pybullet.getQuaternionFromEuler([np.pi, 0, np.pi])
    urdf = 'robotiq_2f_85/robotiq_2f_85.urdf'
    self.body = pybullet.loadURDF(urdf, pos, rot)
    self.n_joints = pybullet.getNumJoints(self.body)
    self.activated = False

    # Connect gripper base to robot tool.
    pybullet.createConstraint(self.robot, tool, self.body, 0, jointType=pybullet.JOINT_FIXED, jointAxis=[0, 0, 0], parentFramePosition=[0, 0, 0], childFramePosition=[0, 0, -0.07], childFrameOrientation=pybullet.getQuaternionFromEuler([0, 0, np.pi / 2]))

    # Set friction coefficients for gripper fingers.
    for i in range(pybullet.getNumJoints(self.body)):
      pybullet.changeDynamics(self.body, i, lateralFriction=10.0, spinningFriction=1.0, rollingFriction=1.0, frictionAnchor=True)

    # Start thread to handle additional gripper constraints.
    self.motor_joint = 1
    self.constraints_thread = threading.Thread(target=self.step)
    self.constraints_thread.daemon = True
    self.constraints_thread.start()

  # Control joint positions by enforcing hard contraints on gripper behavior.
  # Set one joint as the open/close motor joint (other joints should mimic).
  def step(self):
    while True:
      try:
        currj = [pybullet.getJointState(self.body, i)[0] for i in range(self.n_joints)]
        indj = [6, 3, 8, 5, 10]
        targj = [currj[1], -currj[1], -currj[1], currj[1], currj[1]]
        pybullet.setJointMotorControlArray(self.body, indj, pybullet.POSITION_CONTROL, targj, positionGains=np.ones(5))
      except:
        return
      sleep(0.001)

  # Close gripper fingers.
  def activate(self):
    pybullet.setJointMotorControl2(self.body, self.motor_joint, pybullet.VELOCITY_CONTROL, targetVelocity=1, force=10)
    self.activated = True

  # Open gripper fingers.
  def release(self):
    pybullet.setJointMotorControl2(self.body, self.motor_joint, pybullet.VELOCITY_CONTROL, targetVelocity=-1, force=10)
    self.activated = False

  # If activated and object in gripper: check object contact.
  # If activated and nothing in gripper: check gripper contact.
  # If released: check proximity to surface (disabled).
  def detect_contact(self):
    obj, _, ray_frac = self.check_proximity()
    if self.activated:
      empty = self.grasp_width() < 0.01
      cbody = self.body if empty else obj
      if obj == self.body or obj == 0:
        return False
      return self.external_contact(cbody)
  #   else:
  #     return ray_frac < 0.14 or self.external_contact()

  # Return if body is in contact with something other than gripper
  def external_contact(self, body=None):
    if body is None:
      body = self.body
    pts = pybullet.getContactPoints(bodyA=body)
    pts = [pt for pt in pts if pt[2] != self.body]
    return len(pts) > 0  # pylint: disable=g-explicit-length-test

  def check_grasp(self):
    while self.moving():
      sleep(0.001)
    success = self.grasp_width() > 0.01
    return success

  def grasp_width(self):
    lpad = np.array(pybullet.getLinkState(self.body, 4)[0])
    rpad = np.array(pybullet.getLinkState(self.body, 9)[0])
    dist = np.linalg.norm(lpad - rpad) - 0.047813
    return dist

  def check_proximity(self):
    ee_pos = np.array(pybullet.getLinkState(self.robot, self.tool)[0])
    tool_pos = np.array(pybullet.getLinkState(self.body, 0)[0])
    vec = (tool_pos - ee_pos) / np.linalg.norm((tool_pos - ee_pos))
    ee_targ = ee_pos + vec
    ray_data = pybullet.rayTest(ee_pos, ee_targ)[0]
    obj, link, ray_frac = ray_data[0], ray_data[1], ray_data[2]
    return obj, link, ray_frac

### 6.3 Pick & Place 環境（`PickPlaceEnv`）

OpenAI Gym 風の環境クラスです。主なメソッドは次のとおりです。

| メソッド | 内容 |
|---|---|
| `reset(object_list)` | シミュレーションを初期化し、ロボットと物体をランダムな位置（互いに 15cm 以上離す）に配置 |
| `step(action)` | `{'pick': 座標, 'place': 座標}` を受け取り、**掴んで置く一連の動作** を実行（上空に移動 → 下降 → 掴む → 持ち上げ → 置き場所へ移動 → 接触するまで下降 → 離す → ホーム位置へ） |
| `movep(position)` | 逆運動学（IK）で手先を目標位置に動かす関節角を計算し、`servoj` で関節を動かす |
| `step_sim_and_render()` | 物理シミュレーションを 1 ステップ進め、一定間隔で動画用の画像を保存 |
| `get_camera_image()` / `render_image()` | 仮想カメラで画像を撮影 |
| `get_observation()` | カメラ画像・深度から上から見た画像（heightmap / colormap）を作る |
| `get_obj_pos(name)` | 物体名（または「top left corner」など）から 3 次元座標を返す |

**注意**：論文の実機実験では画像認識で物体の位置を推定していますが、このシミュレーションでは **シミュレータが知っている正確な位置（ground truth）** を使っています。

In [ ]:
# Gym-style environment code

class PickPlaceEnv():

  def __init__(self, render=False, high_res=False, high_frame_rate=False):
    self.dt = 1/480
    self.sim_step = 0

    # Configure and start PyBullet.
    # python3 -m pybullet_utils.runServer
    # pybullet.connect(pybullet.SHARED_MEMORY)  # pybullet.GUI for local GUI.
    pybullet.connect(pybullet.DIRECT)  # pybullet.GUI for local GUI.
    pybullet.configureDebugVisualizer(pybullet.COV_ENABLE_GUI, 0)
    pybullet.setPhysicsEngineParameter(enableFileCaching=0)
    assets_path = os.path.dirname(os.path.abspath(""))
    pybullet.setAdditionalSearchPath(assets_path)
    pybullet.setAdditionalSearchPath(pybullet_data.getDataPath())
    pybullet.setTimeStep(self.dt)

    self.home_joints = (np.pi / 2, -np.pi / 2, np.pi / 2, -np.pi / 2, 3 * np.pi / 2, 0)  # Joint angles: (J0, J1, J2, J3, J4, J5).
    self.home_ee_euler = (np.pi, 0, np.pi)  # (RX, RY, RZ) rotation in Euler angles.
    self.ee_link_id = 9  # Link ID of UR5 end effector.
    self.tip_link_id = 10  # Link ID of gripper finger tips.
    self.gripper = None

    self.render = render
    self.high_res = high_res
    self.high_frame_rate = high_frame_rate

  def reset(self, object_list):
    pybullet.resetSimulation(pybullet.RESET_USE_DEFORMABLE_WORLD)
    pybullet.setGravity(0, 0, -9.8)
    self.cache_video = []

    # Temporarily disable rendering to load URDFs faster.
    pybullet.configureDebugVisualizer(pybullet.COV_ENABLE_RENDERING, 0)

    # Add robot.
    pybullet.loadURDF("plane.urdf", [0, 0, -0.001])
    self.robot_id = pybullet.loadURDF("ur5e/ur5e.urdf", [0, 0, 0], flags=pybullet.URDF_USE_MATERIAL_COLORS_FROM_MTL)
    self.ghost_id = pybullet.loadURDF("ur5e/ur5e.urdf", [0, 0, -10])  # For forward kinematics.
    self.joint_ids = [pybullet.getJointInfo(self.robot_id, i) for i in range(pybullet.getNumJoints(self.robot_id))]
    self.joint_ids = [j[0] for j in self.joint_ids if j[2] == pybullet.JOINT_REVOLUTE]

    # Move robot to home configuration.
    for i in range(len(self.joint_ids)):
      pybullet.resetJointState(self.robot_id, self.joint_ids[i], self.home_joints[i])

    # Add gripper.
    if self.gripper is not None:
      while self.gripper.constraints_thread.is_alive():
        self.constraints_thread_active = False
    self.gripper = Robotiq2F85(self.robot_id, self.ee_link_id)
    self.gripper.release()

    # Add workspace.
    plane_shape = pybullet.createCollisionShape(pybullet.GEOM_BOX, halfExtents=[0.3, 0.3, 0.001])
    plane_visual = pybullet.createVisualShape(pybullet.GEOM_BOX, halfExtents=[0.3, 0.3, 0.001])
    plane_id = pybullet.createMultiBody(0, plane_shape, plane_visual, basePosition=[0, -0.5, 0])
    pybullet.changeVisualShape(plane_id, -1, rgbaColor=[0.2, 0.2, 0.2, 1.0])

    # Load objects according to config.
    self.object_list = object_list
    self.obj_name_to_id = {}
    obj_xyz = np.zeros((0, 3))
    for obj_name in object_list:
      if ('block' in obj_name) or ('bowl' in obj_name):

        # Get random position 15cm+ from other objects.
        while True:
          rand_x = np.random.uniform(BOUNDS[0, 0] + 0.1, BOUNDS[0, 1] - 0.1)
          rand_y = np.random.uniform(BOUNDS[1, 0] + 0.1, BOUNDS[1, 1] - 0.1)
          rand_xyz = np.float32([rand_x, rand_y, 0.03]).reshape(1, 3)
          if len(obj_xyz) == 0:
            obj_xyz = np.concatenate((obj_xyz, rand_xyz), axis=0)
            break
          else:
            nn_dist = np.min(np.linalg.norm(obj_xyz - rand_xyz, axis=1)).squeeze()
            if nn_dist > 0.15:
              obj_xyz = np.concatenate((obj_xyz, rand_xyz), axis=0)
              break
        
        object_color = COLORS[obj_name.split(' ')[0]]
        object_type = obj_name.split(' ')[1]
        object_position = rand_xyz.squeeze()
        if object_type == 'block':
          object_shape = pybullet.createCollisionShape(pybullet.GEOM_BOX, halfExtents=[0.02, 0.02, 0.02])
          object_visual = pybullet.createVisualShape(pybullet.GEOM_BOX, halfExtents=[0.02, 0.02, 0.02])
          object_id = pybullet.createMultiBody(0.01, object_shape, object_visual, basePosition=object_position)
        elif object_type == 'bowl':
          object_position[2] = 0
          object_id = pybullet.loadURDF("bowl/bowl.urdf", object_position, useFixedBase=1)
        pybullet.changeVisualShape(object_id, -1, rgbaColor=object_color)
        self.obj_name_to_id[obj_name] = object_id
    

    # Re-enable rendering.
    pybullet.configureDebugVisualizer(pybullet.COV_ENABLE_RENDERING, 1)

    for _ in range(200):
      pybullet.stepSimulation()

    # record object positions at reset
    self.init_pos = {name: self.get_obj_pos(name) for name in object_list}

    return self.get_observation()

  def servoj(self, joints):
    """Move to target joint positions with position control."""
    pybullet.setJointMotorControlArray(
      bodyIndex=self.robot_id,
      jointIndices=self.joint_ids,
      controlMode=pybullet.POSITION_CONTROL,
      targetPositions=joints,
      positionGains=[0.01]*6)
  
  def movep(self, position):
    """Move to target end effector position."""
    joints = pybullet.calculateInverseKinematics(
        bodyUniqueId=self.robot_id,
        endEffectorLinkIndex=self.tip_link_id,
        targetPosition=position,
        targetOrientation=pybullet.getQuaternionFromEuler(self.home_ee_euler),
        maxNumIterations=100)
    self.servoj(joints)

  def get_ee_pos(self):
    ee_xyz = np.float32(pybullet.getLinkState(self.robot_id, self.tip_link_id)[0])
    return ee_xyz

  def step(self, action=None):
    """Do pick and place motion primitive."""
    pick_pos, place_pos = action['pick'].copy(), action['place'].copy()

    # Set fixed primitive z-heights.
    hover_xyz = np.float32([pick_pos[0], pick_pos[1], 0.2])
    if pick_pos.shape[-1] == 2:
      pick_xyz = np.append(pick_pos, 0.025)
    else:
      pick_xyz = pick_pos
      pick_xyz[2] = 0.025
    if place_pos.shape[-1] == 2:
      place_xyz = np.append(place_pos, 0.15)
    else:
      place_xyz = place_pos
      place_xyz[2] = 0.15

    # Move to object.
    ee_xyz = self.get_ee_pos()
    while np.linalg.norm(hover_xyz - ee_xyz) > 0.01:
      self.movep(hover_xyz)
      self.step_sim_and_render()
      ee_xyz = self.get_ee_pos()

    while np.linalg.norm(pick_xyz - ee_xyz) > 0.01:
      self.movep(pick_xyz)
      self.step_sim_and_render()
      ee_xyz = self.get_ee_pos()

    # Pick up object.
    self.gripper.activate()
    for _ in range(240):
      self.step_sim_and_render()
    while np.linalg.norm(hover_xyz - ee_xyz) > 0.01:
      self.movep(hover_xyz)
      self.step_sim_and_render()
      ee_xyz = self.get_ee_pos()
    
    for _ in range(50):
      self.step_sim_and_render()
    
    # Move to place location.
    while np.linalg.norm(place_xyz - ee_xyz) > 0.01:
      self.movep(place_xyz)
      self.step_sim_and_render()
      ee_xyz = self.get_ee_pos()

    # Place down object.
    while (not self.gripper.detect_contact()) and (place_xyz[2] > 0.03):
      place_xyz[2] -= 0.001
      self.movep(place_xyz)
      for _ in range(3):
        self.step_sim_and_render()
    self.gripper.release()
    for _ in range(240):
      self.step_sim_and_render()
    place_xyz[2] = 0.2
    ee_xyz = self.get_ee_pos()
    while np.linalg.norm(place_xyz - ee_xyz) > 0.01:
      self.movep(place_xyz)
      self.step_sim_and_render()
      ee_xyz = self.get_ee_pos()
    place_xyz = np.float32([0, -0.5, 0.2])
    while np.linalg.norm(place_xyz - ee_xyz) > 0.01:
      self.movep(place_xyz)
      self.step_sim_and_render()
      ee_xyz = self.get_ee_pos()

    observation = self.get_observation()
    reward = self.get_reward()
    done = False
    info = {}
    return observation, reward, done, info

  def set_alpha_transparency(self, alpha: float) -> None:
    for id in range(20):
      visual_shape_data = pybullet.getVisualShapeData(id)
      for i in range(len(visual_shape_data)):
        object_id, link_index, _, _, _, _, _, rgba_color = visual_shape_data[i]
        rgba_color = list(rgba_color[0:3]) +  [alpha]
        pybullet.changeVisualShape(
            self.robot_id, linkIndex=i, rgbaColor=rgba_color)      
        pybullet.changeVisualShape(
            self.gripper.body, linkIndex=i, rgbaColor=rgba_color)

  def step_sim_and_render(self):
    pybullet.stepSimulation()
    self.sim_step += 1

    interval = 40 if self.high_frame_rate else 60
    # Render current image at 8 FPS.
    if self.sim_step % interval == 0 and self.render:
      self.cache_video.append(self.get_camera_image())

  def get_camera_image(self):
    if not self.high_res:
      image_size = (240, 240)
      intrinsics = (120., 0, 120., 0, 120., 120., 0, 0, 1)
    else:
      image_size=(360, 360)
      intrinsics=(180., 0, 180., 0, 180., 180., 0, 0, 1)
    color, _, _, _, _ = env.render_image(image_size, intrinsics)
    return color

  def get_reward(self):
    return None

  def get_observation(self):
    observation = {}

    # Render current image.
    color, depth, position, orientation, intrinsics = self.render_image()

    # Get heightmaps and colormaps.
    points = self.get_pointcloud(depth, intrinsics)
    position = np.float32(position).reshape(3, 1)
    rotation = pybullet.getMatrixFromQuaternion(orientation)
    rotation = np.float32(rotation).reshape(3, 3)
    transform = np.eye(4)
    transform[:3, :] = np.hstack((rotation, position))
    points = self.transform_pointcloud(points, transform)
    heightmap, colormap, xyzmap = self.get_heightmap(points, color, BOUNDS, PIXEL_SIZE)

    observation["image"] = colormap
    observation["xyzmap"] = xyzmap

    return observation

  def render_image(self, image_size=(720, 720), intrinsics=(360., 0, 360., 0, 360., 360., 0, 0, 1)):

    # Camera parameters.
    position = (0, -0.85, 0.4)
    orientation = (np.pi / 4 + np.pi / 48, np.pi, np.pi)
    orientation = pybullet.getQuaternionFromEuler(orientation)
    zrange = (0.01, 10.)
    noise=True

    # OpenGL camera settings.
    lookdir = np.float32([0, 0, 1]).reshape(3, 1)
    updir = np.float32([0, -1, 0]).reshape(3, 1)
    rotation = pybullet.getMatrixFromQuaternion(orientation)
    rotm = np.float32(rotation).reshape(3, 3)
    lookdir = (rotm @ lookdir).reshape(-1)
    updir = (rotm @ updir).reshape(-1)
    lookat = position + lookdir
    focal_len = intrinsics[0]
    znear, zfar = (0.01, 10.)
    viewm = pybullet.computeViewMatrix(position, lookat, updir)
    fovh = (image_size[0] / 2) / focal_len
    fovh = 180 * np.arctan(fovh) * 2 / np.pi

    # Notes: 1) FOV is vertical FOV 2) aspect must be float
    aspect_ratio = image_size[1] / image_size[0]
    projm = pybullet.computeProjectionMatrixFOV(fovh, aspect_ratio, znear, zfar)

    # Render with OpenGL camera settings.
    _, _, color, depth, segm = pybullet.getCameraImage(
        width=image_size[1],
        height=image_size[0],
        viewMatrix=viewm,
        projectionMatrix=projm,
        shadow=1,
        flags=pybullet.ER_SEGMENTATION_MASK_OBJECT_AND_LINKINDEX,
        renderer=pybullet.ER_BULLET_HARDWARE_OPENGL)

    # Get color image.
    color_image_size = (image_size[0], image_size[1], 4)
    color = np.array(color, dtype=np.uint8).reshape(color_image_size)
    color = color[:, :, :3]  # remove alpha channel
    if noise:
      color = np.int32(color)
      color += np.int32(np.random.normal(0, 3, color.shape))
      color = np.uint8(np.clip(color, 0, 255))

    # Get depth image.
    depth_image_size = (image_size[0], image_size[1])
    zbuffer = np.float32(depth).reshape(depth_image_size)
    depth = (zfar + znear - (2 * zbuffer - 1) * (zfar - znear))
    depth = (2 * znear * zfar) / depth
    if noise:
      depth += np.random.normal(0, 0.003, depth.shape)

    intrinsics = np.float32(intrinsics).reshape(3, 3)
    return color, depth, position, orientation, intrinsics

  def get_pointcloud(self, depth, intrinsics):
    """Get 3D pointcloud from perspective depth image.
    Args:
      depth: HxW float array of perspective depth in meters.
      intrinsics: 3x3 float array of camera intrinsics matrix.
    Returns:
      points: HxWx3 float array of 3D points in camera coordinates.
    """
    height, width = depth.shape
    xlin = np.linspace(0, width - 1, width)
    ylin = np.linspace(0, height - 1, height)
    px, py = np.meshgrid(xlin, ylin)
    px = (px - intrinsics[0, 2]) * (depth / intrinsics[0, 0])
    py = (py - intrinsics[1, 2]) * (depth / intrinsics[1, 1])
    points = np.float32([px, py, depth]).transpose(1, 2, 0)
    return points

  def transform_pointcloud(self, points, transform):
    """Apply rigid transformation to 3D pointcloud.
    Args:
      points: HxWx3 float array of 3D points in camera coordinates.
      transform: 4x4 float array representing a rigid transformation matrix.
    Returns:
      points: HxWx3 float array of transformed 3D points.
    """
    padding = ((0, 0), (0, 0), (0, 1))
    homogen_points = np.pad(points.copy(), padding,
                            'constant', constant_values=1)
    for i in range(3):
      points[Ellipsis, i] = np.sum(transform[i, :] * homogen_points, axis=-1)
    return points

  def get_heightmap(self, points, colors, bounds, pixel_size):
    """Get top-down (z-axis) orthographic heightmap image from 3D pointcloud.
    Args:
      points: HxWx3 float array of 3D points in world coordinates.
      colors: HxWx3 uint8 array of values in range 0-255 aligned with points.
      bounds: 3x2 float array of values (rows: X,Y,Z; columns: min,max) defining
        region in 3D space to generate heightmap in world coordinates.
      pixel_size: float defining size of each pixel in meters.
    Returns:
      heightmap: HxW float array of height (from lower z-bound) in meters.
      colormap: HxWx3 uint8 array of backprojected color aligned with heightmap.
      xyzmap: HxWx3 float array of XYZ points in world coordinates.
    """
    width = int(np.round((bounds[0, 1] - bounds[0, 0]) / pixel_size))
    height = int(np.round((bounds[1, 1] - bounds[1, 0]) / pixel_size))
    heightmap = np.zeros((height, width), dtype=np.float32)
    colormap = np.zeros((height, width, colors.shape[-1]), dtype=np.uint8)
    xyzmap = np.zeros((height, width, 3), dtype=np.float32)

    # Filter out 3D points that are outside of the predefined bounds.
    ix = (points[Ellipsis, 0] >= bounds[0, 0]) & (points[Ellipsis, 0] < bounds[0, 1])
    iy = (points[Ellipsis, 1] >= bounds[1, 0]) & (points[Ellipsis, 1] < bounds[1, 1])
    iz = (points[Ellipsis, 2] >= bounds[2, 0]) & (points[Ellipsis, 2] < bounds[2, 1])
    valid = ix & iy & iz
    points = points[valid]
    colors = colors[valid]

    # Sort 3D points by z-value, which works with array assignment to simulate
    # z-buffering for rendering the heightmap image.
    iz = np.argsort(points[:, -1])
    points, colors = points[iz], colors[iz]
    px = np.int32(np.floor((points[:, 0] - bounds[0, 0]) / pixel_size))
    py = np.int32(np.floor((points[:, 1] - bounds[1, 0]) / pixel_size))
    px = np.clip(px, 0, width - 1)
    py = np.clip(py, 0, height - 1)
    heightmap[py, px] = points[:, 2] - bounds[2, 0]
    for c in range(colors.shape[-1]):
      colormap[py, px, c] = colors[:, c]
      xyzmap[py, px, c] = points[:, c]
    colormap = colormap[::-1, :, :]  # Flip up-down.
    xv, yv = np.meshgrid(np.linspace(BOUNDS[0, 0], BOUNDS[0, 1], height),
                         np.linspace(BOUNDS[1, 0], BOUNDS[1, 1], width))
    xyzmap[:, :, 0] = xv
    xyzmap[:, :, 1] = yv
    xyzmap = xyzmap[::-1, :, :]  # Flip up-down.
    heightmap = heightmap[::-1, :]  # Flip up-down.
    return heightmap, colormap, xyzmap
  
  def on_top_of(self, obj_a, obj_b):
    """
    check if obj_a is on top of obj_b
    condition 1: l2 distance on xy plane is less than a threshold
    condition 2: obj_a is higher than obj_b
    """
    obj_a_pos = self.get_obj_pos(obj_a)
    obj_b_pos = self.get_obj_pos(obj_b)
    xy_dist = np.linalg.norm(obj_a_pos[:2] - obj_b_pos[:2])
    if obj_b in CORNER_POS:
      is_near = xy_dist < 0.06
      return is_near
    elif 'bowl' in obj_b:
      is_near = xy_dist < 0.06
      is_higher = obj_a_pos[2] > obj_b_pos[2]
      return is_near and is_higher
    else:
      is_near = xy_dist < 0.04
      is_higher = obj_a_pos[2] > obj_b_pos[2]
      return is_near and is_higher
  
  def get_obj_id(self, obj_name):
    try:
      if obj_name in self.obj_name_to_id:
        obj_id = self.obj_name_to_id[obj_name]
      else:
        obj_name = obj_name.replace('circle', 'bowl').replace('square', 'block').replace('small', '').strip()
        obj_id = self.obj_name_to_id[obj_name]
    except:
      print(f'requested_name="{obj_name}"')
      print(f'available_objects_and_id="{self.obj_name_to_id}')
    return obj_id
  
  def get_obj_pos(self, obj_name):
    obj_name = obj_name.replace('the', '').replace('_', ' ').strip()
    if obj_name in CORNER_POS:
      position = np.float32(np.array(CORNER_POS[obj_name]))
    else:
      pick_id = self.get_obj_id(obj_name)
      pose = pybullet.getBasePositionAndOrientation(pick_id)
      position = np.float32(pose[0])
    return position
  
  def get_bounding_box(self, obj_name):
    obj_id = self.get_obj_id(obj_name)
    return pybullet.getAABB(obj_id)

## 7. LMP から使うロボット API（`LMP_wrapper`）

LLM が生成したコードは、シミュレータを直接操作するのではなく、この **ラッパークラスのメソッドを通じて** ロボットを動かします。LLM に見せる API を小さく分かりやすく保つための層です。

| メソッド | 生成コードでの使われ方 |
|---|---|
| `get_obj_names()` | シーンにある物体名の一覧 |
| `get_obj_pos(name)` | 物体の xy 座標（2 次元） |
| `put_first_on_second(a, b)` | 物体 a を掴んで、物体 b の上（または座標 b）に置く。**唯一の動作 API** |
| `denormalize_xy([u, v])` | 0〜1 に正規化された座標を実際の座標に変換（`[0.5, 0.5]` が中央、`[1, 1]` が右上） |
| `get_corner_name(pos)` / `get_side_name(pos)` | 座標から最も近い角・辺の名前を返す（ロボットの発話用） |
| `get_bbox(name)` | 物体の外接直方体 |
| `is_obj_visible(name)` / `get_color(name)` | 物体が存在するか / 物体の色 |

※ 元のコードにあった `get_corner_name` の `'botom right corner'` というタイポは `'bottom right corner'` に修正しています。

In [ ]:
class LMP_wrapper():

  def __init__(self, env, cfg, render=False):
    self.env = env
    self._cfg = cfg
    self.object_names = list(self._cfg['env']['init_objs'])
    
    self._min_xy = np.array(self._cfg['env']['coords']['bottom_left'])
    self._max_xy = np.array(self._cfg['env']['coords']['top_right'])
    self._range_xy = self._max_xy - self._min_xy

    self._table_z = self._cfg['env']['coords']['table_z']
    self.render = render

  def is_obj_visible(self, obj_name):
    return obj_name in self.object_names

  def get_obj_names(self):
    return self.object_names[::]

  def denormalize_xy(self, pos_normalized):
    return pos_normalized * self._range_xy + self._min_xy

  def get_corner_positions(self):
    unit_square = box(0, 0, 1, 1)
    normalized_corners = np.array(list(unit_square.exterior.coords))[:4]
    corners = np.array(([self.denormalize_xy(corner) for corner in normalized_corners]))
    return corners

  def get_side_positions(self):
    side_xs = np.array([0, 0.5, 0.5, 1])
    side_ys = np.array([0.5, 0, 1, 0.5])
    normalized_side_positions = np.c_[side_xs, side_ys]
    side_positions = np.array(([self.denormalize_xy(corner) for corner in normalized_side_positions]))
    return side_positions

  def get_obj_pos(self, obj_name):
    # return the xy position of the object in robot base frame
    return self.env.get_obj_pos(obj_name)[:2]

  def get_obj_position_np(self, obj_name):
    return self.get_pos(obj_name)

  def get_bbox(self, obj_name):
    # return the axis-aligned object bounding box in robot base frame (not in pixels)
    # the format is (min_x, min_y, max_x, max_y)
    bbox = self.env.get_bounding_box(obj_name)
    return bbox

  def get_color(self, obj_name):
    for color, rgb in COLORS.items():
      if color in obj_name:
        return rgb

  def pick_place(self, pick_pos, place_pos):
    pick_pos_xyz = np.r_[pick_pos, [self._table_z]]
    place_pos_xyz = np.r_[place_pos, [self._table_z]]
    pass

  def put_first_on_second(self, arg1, arg2):
    # put the object with obj_name on top of target
    # target can either be another object name, or it can be an x-y position in robot base frame
    pick_pos = self.get_obj_pos(arg1) if isinstance(arg1, str) else arg1
    place_pos = self.get_obj_pos(arg2) if isinstance(arg2, str) else arg2
    self.env.step(action={'pick': pick_pos, 'place': place_pos})

  def get_robot_pos(self):
    # return robot end-effector xy position in robot base frame
    return self.env.get_ee_pos()

  def goto_pos(self, position_xy):
    # move the robot end-effector to the desired xy position while maintaining same z
    ee_xyz = self.env.get_ee_pos()
    position_xyz = np.concatenate([position_xy, ee_xyz[-1]])
    while np.linalg.norm(position_xyz - ee_xyz) > 0.01:
      self.env.movep(position_xyz)
      self.env.step_sim_and_render()
      ee_xyz = self.env.get_ee_pos()

  def follow_traj(self, traj):
    for pos in traj:
      self.goto_pos(pos)

  def get_corner_positions(self):
    normalized_corners = np.array([
        [0, 1],
        [1, 1],
        [0, 0],
        [1, 0]
    ])
    return np.array(([self.denormalize_xy(corner) for corner in normalized_corners]))

  def get_side_positions(self):
    normalized_sides = np.array([
        [0.5, 1],
        [1, 0.5],
        [0.5, 0],
        [0, 0.5]
    ])
    return np.array(([self.denormalize_xy(side) for side in normalized_sides]))

  def get_corner_name(self, pos):
    corner_positions = self.get_corner_positions()
    corner_idx = np.argmin(np.linalg.norm(corner_positions - pos, axis=1))
    return ['top left corner', 'top right corner', 'bottom left corner', 'bottom right corner'][corner_idx]

  def get_side_name(self, pos):
    side_positions = self.get_side_positions()
    side_idx = np.argmin(np.linalg.norm(side_positions - pos, axis=1))
    return ['top side', 'right side', 'bottom side', 'left side'][side_idx]

## 8. LMP のプロンプト（few-shot 例）

Code as Policies では、モデルを追加学習（ファインチューニング）せず、**プロンプトに例を並べるだけ** でロボット制御コードを書かせます。各プロンプトは次の形式になっています。

```python
objects = ['yellow block', 'green block', 'yellow bowl', ...]   # ← 文脈（シーンの物体）
# place the yellow block on the yellow bowl.                    # ← 指示（コメント）
say('Ok - putting the yellow block on the yellow bowl')        # ← 答えのコード
put_first_on_second('yellow block', 'yellow bowl')
```

LLM はこのパターンを読み取り、最後に置かれた新しい指示コメントに対する「答えのコード」を続けて書きます。

**プロンプトに関する重要なポイント**

- プロンプト冒頭の `import` 文は実行されません。LLM に「どんな関数が使えるか」を伝えるためだけのものです
- 例の中で使われている `get_obj_positions_np`、`get_closest_idx`、`make_line`、`interpolate_pts_on_line` などは **どこにも実装されていません**。実行時に LLM が使うと、前述の `LMPFGen` が自動で実装します
- 例はすべて英語なので、指示も英語で書くのが最も安定します（Gemini は多言語に強いので、日本語の指示でもある程度動きます）

### 8.1 `tabletop_ui`：ユーザーの指示を最初に受け取る LMP

高レベルの指示を、`say()`（ロボットの発話）・`put_first_on_second()`・下位の LMP（`parse_obj_name` など）の呼び出しに分解します。例を見ると、

- 単純な移動（「黄色いブロックを黄色いボウルに置いて」）
- 質問への回答（「どのブロックを動かした？」）
- 曖昧な表現の解釈（「バナナ色の小さいもの」→ 黄色いブロック）
- できないことの拒否（「ボウルを半分に切って」→「物を動かすことしかできません」）
- 文脈を使った指示（「元に戻して」）

など、多様な指示への対応方法を示しているのが分かります。

In [ ]:
prompt_tabletop_ui = '''
# Python 2D robot control script
import numpy as np
from env_utils import put_first_on_second, get_obj_pos, get_obj_names, say, get_corner_name, get_side_name, is_obj_visible, stack_objects_in_order
from plan_utils import parse_obj_name, parse_position, parse_question, transform_shape_pts

objects = ['yellow block', 'green block', 'yellow bowl', 'blue block', 'blue bowl', 'green bowl']
# place the yellow block on the yellow bowl.
say('Ok - putting the yellow block on the yellow bowl')
put_first_on_second('yellow block', 'yellow bowl')
objects = ['yellow block', 'green block', 'yellow bowl', 'blue block', 'blue bowl', 'green bowl']
# which block did you move.
say('I moved the yellow block')
objects = ['yellow block', 'green block', 'yellow bowl', 'blue block', 'blue bowl', 'green bowl']
# move the green block to the top right corner.
say('Got it - putting the green block on the top right corner')
corner_pos = parse_position('top right corner')
put_first_on_second('green block', corner_pos)
objects = ['yellow block', 'green block', 'yellow bowl', 'blue block', 'blue bowl', 'green bowl']
# stack the blue bowl on the yellow bowl on the green block.
order_bottom_to_top = ['green block', 'yellow block', 'blue bowl']
say(f'Sure - stacking from top to bottom: {", ".join(order_bottom_to_top)}')
stack_objects_in_order(object_names=order_bottom_to_top)
objects = ['cyan block', 'white block', 'cyan bowl', 'blue block', 'blue bowl', 'white bowl']
# move the cyan block into its corresponding bowl.
matches = {'cyan block': 'cyan bowl'}
say('Got it - placing the cyan block on the cyan bowl')
for first, second in matches.items():
  put_first_on_second(first, get_obj_pos(second))
objects = ['cyan block', 'white block', 'cyan bowl', 'blue block', 'blue bowl', 'white bowl']
# make a line of blocks on the right side.
say('No problem! Making a line of blocks on the right side')
block_names = parse_obj_name('the blocks', f'objects = {get_obj_names()}')
line_pts = parse_position(f'a 30cm vertical line on the right with {len(block_names)} points')
for block_name, pt in zip(block_names, line_pts):
  put_first_on_second(block_name, pt)
objects = ['yellow block', 'red block', 'yellow bowl', 'gray block', 'gray bowl', 'red bowl']
# put the small banana colored thing in between the blue bowl and green block.
say('Sure thing - putting the yellow block between the blue bowl and the green block')
target_pos = parse_position('a point in the middle betweeen the blue bowl and the green block')
put_first_on_second('yellow block', target_pos)
objects = ['yellow block', 'red block', 'yellow bowl', 'gray block', 'gray bowl', 'red bowl']
# can you cut the bowls in half.
say('no, I can only move objects around')
objects = ['yellow block', 'green block', 'yellow bowl', 'gray block', 'gray bowl', 'green bowl']
# stack the blocks on the right side with the gray one on the bottom.
say('Ok. stacking the blocks on the right side with the gray block on the bottom')
right_side = parse_position('the right side')
put_first_on_second('gray block', right_side)
order_bottom_to_top = ['gray block', 'green block', 'yellow block']
stack_objects_in_order(object_names=order_bottom_to_top)
objects = ['yellow block', 'green block', 'yellow bowl', 'blue block', 'blue bowl', 'green bowl']
# hide the blue bowl.
bowl_name = np.random.choice(['yellow bowl', 'green bowl'])
say(f'Sounds good! Hiding the blue bowl under the {bowl_name}')
put_first_on_second(bowl_name, 'blue bowl')
objects = ['pink block', 'green block', 'pink bowl', 'blue block', 'blue bowl', 'green bowl']
# move the grass-colored bowl to the left.
say('Sure - moving the green bowl left by 10 centimeters')
left_pos = parse_position('a point 10cm left of the green bowl')
put_first_on_second('green bowl', left_pos)
objects = ['pink block', 'green block', 'pink bowl', 'blue block', 'blue bowl', 'green bowl']
# why did you move the red bowl.
say(f'I did not move the red bowl')
objects = ['pink block', 'green block', 'pink bowl', 'blue block', 'blue bowl', 'green bowl']
# undo that.
say('Sure - moving the green bowl right by 10 centimeters')
left_pos = parse_position('a point 10cm right of the green bowl')
put_first_on_second('green bowl', left_pos)
objects = ['brown bowl', 'green block', 'brown block', 'green bowl', 'blue bowl', 'blue block']
# place the top most block to the corner closest to the bottom most block.
top_block_name = parse_obj_name('top most block', f'objects = {get_obj_names()}')
bottom_block_name = parse_obj_name('bottom most block', f'objects = {get_obj_names()}')
closest_corner_pos = parse_position(f'the corner closest to the {bottom_block_name}', f'objects = {get_obj_names()}')
say(f'Putting the {top_block_name} on the {get_corner_name(closest_corner_pos)}')
put_first_on_second(top_block_name, closest_corner_pos)
objects = ['brown bowl', 'green block', 'brown block', 'green bowl', 'blue bowl', 'blue block']
# move the brown bowl to the side closest to the green block.
closest_side_position = parse_position('the side closest to the green block')
say(f'Got it - putting the brown bowl on the {get_side_name(closest_side_position)}')
put_first_on_second('brown bowl', closest_side_position)
objects = ['brown bowl', 'green block', 'brown block', 'green bowl', 'blue bowl', 'blue block']
# place the green block to the right of the bowl that has the blue block.
bowl_name = parse_obj_name('the bowl that has the blue block', f'objects = {get_obj_names()}')
if bowl_name:
  target_pos = parse_position(f'a point 10cm to the right of the {bowl_name}')
  say(f'No problem - placing the green block to the right of the {bowl_name}')
  put_first_on_second('green block', target_pos)
else:
  say('There are no bowls that has the blue block')
objects = ['brown bowl', 'green block', 'brown block', 'green bowl', 'blue bowl', 'blue block']
# move the other blocks to the bottom corners.
block_names = parse_obj_name('blocks other than the blue block', f'objects = {get_obj_names()}')
corners = parse_position('the bottom corners')
for block_name, pos in zip(block_names, corners):
  put_first_on_second(block_name, pos)
objects = ['pink block', 'gray block', 'orange block']
# move the pinkish colored block on the bottom side.
say('Ok - putting the pink block on the bottom side')
bottom_side_pos = parse_position('the bottom side')
put_first_on_second('pink block', bottom_side_pos)
objects = ['yellow bowl', 'blue block', 'yellow block', 'blue bowl']
# is the blue block to the right of the yellow bowl?
if parse_question('is the blue block to the right of the yellow bowl?', f'objects = {get_obj_names()}'):
  say('yes, there is a blue block to the right of the yellow bow')
else:
  say('no, there is\'t a blue block to the right of the yellow bow')
objects = ['yellow bowl', 'blue block', 'yellow block', 'blue bowl']
# how many yellow objects are there?
n_yellow_objs = parse_question('how many yellow objects are there', f'objects = {get_obj_names()}')
say(f'there are {n_yellow_objs} yellow object')
objects = ['pink block', 'green block', 'pink bowl', 'blue block', 'blue bowl', 'green bowl']
# move the left most block to the green bowl.
left_block_name = parse_obj_name('left most block', f'objects = {get_obj_names()}')
say(f'Moving the {left_block_name} on the green bowl')
put_first_on_second(left_block_name, 'green bowl')
objects = ['pink block', 'green block', 'pink bowl', 'blue block', 'blue bowl', 'green bowl']
# move the other blocks to different corners.
block_names = parse_obj_name(f'blocks other than the {left_block_name}', f'objects = {get_obj_names()}')
corners = parse_position('the corners')
say(f'Ok - moving the other {len(block_names)} blocks to different corners')
for block_name, pos in zip(block_names, corners):
  put_first_on_second(block_name, pos)
objects = ['pink block', 'green block', 'pink bowl', 'blue block', 'blue bowl', 'green bowl']
# is the pink block on the green bowl.
if parse_question('is the pink block on the green bowl', f'objects = {get_obj_names()}'):
  say('Yes - the pink block is on the green bowl.')
else:
  say('No - the pink block is not on the green bowl.')
objects = ['pink block', 'green block', 'pink bowl', 'blue block', 'blue bowl', 'green bowl']
# what are the blocks left of the green bowl.
left_block_names =  parse_question('what are the blocks left of the green bowl', f'objects = {get_obj_names()}')
if len(left_block_names) > 0:
  say(f'These blocks are left of the green bowl: {", ".join(left_block_names)}')
else:
  say('There are no blocks left of the green bowl')
objects = ['yellow block', 'green block', 'yellow bowl', 'blue block', 'blue bowl', 'green bowl']
# imagine that the bowls are different biomes on earth and imagine that the blocks are parts of a building.
say('ok')
objects = ['yellow block', 'green block', 'yellow bowl', 'blue block', 'blue bowl', 'green bowl']
# now build a tower in the grasslands.
order_bottom_to_top = ['green bowl', 'blue block', 'green block', 'yellow block']
say('stacking the blocks on the green bowl')
stack_objects_in_order(object_names=order_bottom_to_top)
objects = ['yellow block', 'green block', 'yellow bowl', 'gray block', 'gray bowl', 'green bowl']
# show me what happens when the desert gets flooded by the ocean.
say('putting the yellow bowl on the blue bowl')
put_first_on_second('yellow bowl', 'blue bowl')
objects = ['pink block', 'gray block', 'orange block']
# move all blocks 5cm toward the top.
say('Ok - moving all blocks 5cm toward the top')
block_names = parse_obj_name('the blocks', f'objects = {get_obj_names()}')
for block_name in block_names:
  target_pos = parse_position(f'a point 5cm above the {block_name}')
  put_first_on_second(block_name, target_pos)
objects = ['cyan block', 'white block', 'purple bowl', 'blue block', 'blue bowl', 'white bowl']
# make a triangle of blocks in the middle.
block_names = parse_obj_name('the blocks', f'objects = {get_obj_names()}')
triangle_pts = parse_position(f'a triangle with size 10cm around the middle with {len(block_names)} points')
say('Making a triangle of blocks around the middle of the workspace')
for block_name, pt in zip(block_names, triangle_pts):
  put_first_on_second(block_name, pt)
objects = ['cyan block', 'white block', 'purple bowl', 'blue block', 'blue bowl', 'white bowl']
# make the triangle smaller.
triangle_pts = transform_shape_pts('scale it by 0.5x', shape_pts=triangle_pts)
say('Making the triangle smaller')
block_names = parse_obj_name('the blocks', f'objects = {get_obj_names()}')
for block_name, pt in zip(block_names, triangle_pts):
  put_first_on_second(block_name, pt)
objects = ['brown bowl', 'red block', 'brown block', 'red bowl', 'pink bowl', 'pink block']
# put the red block on the farthest bowl.
farthest_bowl_name = parse_obj_name('the bowl farthest from the red block', f'objects = {get_obj_names()}')
say(f'Putting the red block on the {farthest_bowl_name}')
put_first_on_second('red block', farthest_bowl_name)
'''.strip()

### 8.2 `parse_obj_name`：自然言語の表現から物体名を特定する

「一番左のブロック」「紫のボウルに最も近いブロック」「茶色のもの」といった表現を、実際の物体名（または物体名のリスト）に変換し、`ret_val` に入れて返します。位置関係は numpy で座標を比較して求めています。

In [ ]:
prompt_parse_obj_name = '''
import numpy as np
from env_utils import get_obj_pos, parse_position
from utils import get_obj_positions_np

objects = ['blue block', 'cyan block', 'purple bowl', 'gray bowl', 'brown bowl', 'pink block', 'purple block']
# the block closest to the purple bowl.
block_names = ['blue block', 'cyan block', 'purple block']
block_positions = get_obj_positions_np(block_names)
closest_block_idx = get_closest_idx(points=block_positions, point=get_obj_pos('purple bowl'))
closest_block_name = block_names[closest_block_idx]
ret_val = closest_block_name
objects = ['brown bowl', 'banana', 'brown block', 'apple', 'blue bowl', 'blue block']
# the blocks.
ret_val = ['brown block', 'blue block']
objects = ['brown bowl', 'banana', 'brown block', 'apple', 'blue bowl', 'blue block']
# the brown objects.
ret_val = ['brown bowl', 'brown block']
objects = ['brown bowl', 'banana', 'brown block', 'apple', 'blue bowl', 'blue block']
# a fruit that's not the apple
fruit_names = ['banana', 'apple']
for fruit_name in fruit_names:
    if fruit_name != 'apple':
        ret_val = fruit_name
objects = ['blue block', 'cyan block', 'purple bowl', 'brown bowl', 'purple block']
# blocks above the brown bowl.
block_names = ['blue block', 'cyan block', 'purple block']
brown_bowl_pos = get_obj_pos('brown bowl')
use_block_names = []
for block_name in block_names:
    if get_obj_pos(block_name)[1] > brown_bowl_pos[1]:
        use_block_names.append(block_name)
ret_val = use_block_names
objects = ['blue block', 'cyan block', 'purple bowl', 'brown bowl', 'purple block']
# the blue block.
ret_val = 'blue block'
objects = ['blue block', 'cyan block', 'purple bowl', 'brown bowl', 'purple block']
# the block closest to the bottom right corner.
corner_pos = parse_position('bottom right corner')
block_names = ['blue block', 'cyan block', 'purple block']
block_positions = get_obj_positions_np(block_names)
closest_block_idx = get_closest_idx(points=block_positions, point=corner_pos)
closest_block_name = block_names[closest_block_idx]
ret_val = closest_block_name
objects = ['brown bowl', 'green block', 'brown block', 'green bowl', 'blue bowl', 'blue block']
# the left most block.
block_names = ['green block', 'brown block', 'blue block']
block_positions = get_obj_positions_np(block_names)
left_block_idx = np.argsort(block_positions[:, 0])[0]
left_block_name = block_names[left_block_idx]
ret_val = left_block_name
objects = ['brown bowl', 'green block', 'brown block', 'green bowl', 'blue bowl', 'blue block']
# the bowl on near the top.
bowl_names = ['brown bowl', 'green bowl', 'blue bowl']
bowl_positions = get_obj_positions_np(bowl_names)
top_bowl_idx = np.argsort(bowl_positions[:, 1])[-1]
top_bowl_name = bowl_names[top_bowl_idx]
ret_val = top_bowl_name
objects = ['yellow bowl', 'purple block', 'yellow block', 'purple bowl', 'pink bowl', 'pink block']
# the third bowl from the right.
bowl_names = ['yellow bowl', 'purple bowl', 'pink bowl']
bowl_positions = get_obj_positions_np(bowl_names)
bowl_idx = np.argsort(bowl_positions[:, 0])[-3]
bowl_name = bowl_names[bowl_idx]
ret_val = bowl_name
'''.strip()

### 8.3 `parse_position`：自然言語の表現から座標を計算する

「中央に 30cm の横線を 3 点で」「緑のボウルの 10cm 左」「下側の辺」などを、座標（または座標のリスト）に変換します。`denormalize_xy` で正規化座標を使ったり、`make_line` や `make_triangle` などの（実行時に生成される）幾何関数を使ったりします。

In [ ]:
prompt_parse_position = '''
import numpy as np
from shapely.geometry import *
from shapely.affinity import *
from env_utils import denormalize_xy, parse_obj_name, get_obj_names, get_obj_pos

# a 30cm horizontal line in the middle with 3 points.
middle_pos = denormalize_xy([0.5, 0.5]) 
start_pos = middle_pos + [-0.3/2, 0]
end_pos = middle_pos + [0.3/2, 0]
line = make_line(start=start_pos, end=end_pos)
points = interpolate_pts_on_line(line=line, n=3)
ret_val = points
# a 20cm vertical line near the right with 4 points.
middle_pos = denormalize_xy([1, 0.5]) 
start_pos = middle_pos + [0, -0.2/2]
end_pos = middle_pos + [0, 0.2/2]
line = make_line(start=start_pos, end=end_pos)
points = interpolate_pts_on_line(line=line, n=4)
ret_val = points
# a diagonal line from the top left to the bottom right corner with 5 points.
top_left_corner = denormalize_xy([0, 1])
bottom_right_corner = denormalize_xy([1, 0])
line = make_line(start=top_left_corner, end=bottom_right_corner)
points = interpolate_pts_on_line(line=line, n=5)
ret_val = points
# a triangle with size 10cm with 3 points.
polygon = make_triangle(size=0.1, center=denormalize_xy([0.5, 0.5]))
points = get_points_from_polygon(polygon)
ret_val = points
# the corner closest to the sun colored block.
block_name = parse_obj_name('the sun colored block', f'objects = {get_obj_names()}')
corner_positions = np.array([denormalize_xy(pos) for pos in [[0, 0], [0, 1], [1, 1], [1, 0]]])
closest_corner_pos = get_closest_point(points=corner_positions, point=get_obj_pos(block_name))
ret_val = closest_corner_pos
# the side farthest from the right most bowl.
bowl_name = parse_obj_name('the right most bowl', f'objects = {get_obj_names()}')
side_positions = np.array([denormalize_xy(pos) for pos in [[0.5, 0], [0.5, 1], [1, 0.5], [0, 0.5]]])
farthest_side_pos = get_farthest_point(points=side_positions, point=get_obj_pos(bowl_name))
ret_val = farthest_side_pos
# a point above the third block from the bottom.
block_name = parse_obj_name('the third block from the bottom', f'objects = {get_obj_names()}')
ret_val = get_obj_pos(block_name) + [0.1, 0]
# a point 10cm left of the bowls.
bowl_names = parse_obj_name('the bowls', f'objects = {get_obj_names()}')
bowl_positions = get_all_object_positions_np(obj_names=bowl_names)
left_obj_pos = bowl_positions[np.argmin(bowl_positions[:, 0])] + [-0.1, 0]
ret_val = left_obj_pos
# the bottom side.
bottom_pos = denormalize_xy([0.5, 0])
ret_val = bottom_pos
# the top corners.
top_left_pos = denormalize_xy([0, 1])
top_right_pos = denormalize_xy([1, 1])
ret_val = [top_left_pos, top_right_pos]
'''.strip()

### 8.4 `parse_question`：シーンについての質問に答える

「青いブロックは黄色いボウルの右にある？」「黄色い物体はいくつ？」といった質問の答え（真偽値・数・名前のリスト）を `ret_val` に入れて返します。`tabletop_ui` はこの答えを使って `say()` で返事をします。

In [ ]:
prompt_parse_question = '''
from utils import get_obj_pos, get_obj_names, parse_obj_name, bbox_contains_pt, is_obj_visible

objects = ['yellow bowl', 'blue block', 'yellow block', 'blue bowl', 'fruit', 'green block', 'black bowl']
# is the blue block to the right of the yellow bowl?
ret_val = get_obj_pos('blue block')[0] > get_obj_pos('yellow bowl')[0]
objects = ['yellow bowl', 'blue block', 'yellow block', 'blue bowl', 'fruit', 'green block', 'black bowl']
# how many yellow objects are there?
yellow_object_names = parse_obj_name('the yellow objects', f'objects = {get_obj_names()}')
ret_val = len(yellow_object_names)
objects = ['pink block', 'green block', 'pink bowl', 'blue block', 'blue bowl', 'green bowl']
# is the pink block on the green bowl?
ret_val = bbox_contains_pt(container_name='green bowl', obj_name='pink block')
objects = ['pink block', 'green block', 'pink bowl', 'blue block', 'blue bowl', 'green bowl']
# what are the blocks left of the green bowl?
block_names = parse_obj_name('the blocks', f'objects = {get_obj_names()}')
green_bowl_pos = get_obj_pos('green bowl')
left_block_names = []
for block_name in block_names:
  if get_obj_pos(block_name)[0] < green_bowl_pos[0]:
    left_block_names.append(block_name)
ret_val = left_block_names
objects = ['pink block', 'yellow block', 'pink bowl', 'blue block', 'blue bowl', 'yellow bowl']
# is the sun colored block above the blue bowl?
sun_block_name = parse_obj_name('sun colored block', f'objects = {get_obj_names()}')
sun_block_pos = get_obj_pos(sun_block_name)
blue_bowl_pos = get_obj_pos('blue bowl')
ret_val = sun_block_pos[1] > blue_bowl_pos[1]
objects = ['pink block', 'yellow block', 'pink bowl', 'blue block', 'blue bowl', 'yellow bowl']
# is the green block below the blue bowl?
ret_val = get_obj_pos('green block')[1] < get_obj_pos('blue bowl')[1]
'''.strip()

### 8.5 `transform_shape_pts`：図形（点の集合）を変形する

「1.5 倍にして」「右に 10cm 動かして」「時計回りに 40 度回転して」といった指示で、`shape_pts`（点の配列）を変形した結果を `new_shape_pts` に入れて返します。「三角形に並べて」→「その三角形を小さくして」のような連続した指示で使われます。

In [ ]:
prompt_transform_shape_pts = '''
import numpy as np
from utils import get_obj_pos, get_obj_names, parse_position, parse_obj_name

# make it bigger by 1.5.
new_shape_pts = scale_pts_around_centroid_np(shape_pts, scale_x=1.5, scale_y=1.5)
# move it to the right by 10cm.
new_shape_pts = translate_pts_np(shape_pts, delta=[0.1, 0])
# move it to the top by 20cm.
new_shape_pts = translate_pts_np(shape_pts, delta=[0, 0.2])
# rotate it clockwise by 40 degrees.
new_shape_pts = rotate_pts_around_centroid_np(shape_pts, angle=-np.deg2rad(40))
# rotate by 30 degrees and make it slightly smaller
new_shape_pts = rotate_pts_around_centroid_np(shape_pts, angle=np.deg2rad(30))
new_shape_pts = scale_pts_around_centroid_np(new_shape_pts, scale_x=0.7, scale_y=0.7)
# move it toward the blue block.
block_name = parse_obj_name('the blue block', f'objects = {get_obj_names()}')
block_pos = get_obj_pos(block_name)
mean_delta = np.mean(block_pos - shape_pts, axis=1)
new_shape_pts = translate_pts_np(shape_pts, mean_delta)
'''.strip()

### 8.6 `fgen`：関数を生成するためのプロンプト

`LMPFGen` が使うプロンプトです。`# define function: 戻り値 = 関数名(引数).` という形式のコメントの後に、その関数の実装を書く例が並んでいます。`# example:` の部分は「関数をどう使うか」の例で、LLM に利用可能な API の使い方を伝えています。

In [ ]:
prompt_fgen = '''
import numpy as np
from shapely.geometry import *
from shapely.affinity import *

from env_utils import get_obj_pos, get_obj_names
from ctrl_utils import put_first_on_second

# define function: total = get_total(xs=numbers).
def get_total(xs):
    return np.sum(xs)

# define function: y = eval_line(x, slope, y_intercept=0).
def eval_line(x, slope, y_intercept):
    return x * slope + y_intercept

# define function: pt = get_pt_to_the_left(pt, dist).
def get_pt_to_the_left(pt, dist):
    return pt + [-dist, 0]

# define function: pt = get_pt_to_the_top(pt, dist).
def get_pt_to_the_top(pt, dist):
    return pt + [0, dist]

# define function line = make_line_by_length(length=x).
def make_line_by_length(length):
  line = LineString([[0, 0], [length, 0]])
  return line

# define function: line = make_vertical_line_by_length(length=x).
def make_vertical_line_by_length(length):
  line = make_line_by_length(length)
  vertical_line = rotate(line, 90)
  return vertical_line

# define function: pt = interpolate_line(line, t=0.5).
def interpolate_line(line, t):
  pt = line.interpolate(t, normalized=True)
  return np.array(pt.coords[0])

# example: scale a line by 2.
line = make_line_by_length(1)
new_shape = scale(line, xfact=2, yfact=2)

# example: put object1 on top of object0.
put_first_on_second('object1', 'object0')

# example: get the position of the first object.
obj_names = get_obj_names()
pos_2d = get_obj_pos(obj_names[0])
'''.strip()

## 9. LMP の設定

各 LMP の振る舞いを辞書で設定します。元の `'engine': 'text-davinci-003'` を `'model': MODEL_NAME` に置き換えています。

| キー | 意味 |
|---|---|
| `prompt_text` | few-shot 例のプロンプト |
| `model` | 使用する Gemini モデル |
| `max_tokens` | 生成するトークン数の上限 |
| `temperature` | ランダム性（0 = 最も決定的） |
| `query_prefix` / `query_suffix` | 指示をコメント化するための前後の文字列（`# ` と `.`） |
| `stop` | この文字列で始まる行が出たら生成を打ち切る（次の例に進もうとしたら止める） |
| `maintain_session` | 過去の実行履歴をプロンプトに含めるか（`tabletop_ui` のみ True） |
| `debug_mode` | True にすると生成だけしてコードを実行しない |
| `include_context` | `objects = [...]` などの文脈も一緒に実行するか |
| `has_return` / `return_val_name` | 値を返す LMP か / 返す変数の名前 |

`lmp_tabletop_coords` は作業台の角・辺・中央の座標で、`LMP_wrapper` の `denormalize_xy` の基準になります。

In [ ]:
cfg_tabletop = {
  'lmps': {
    'tabletop_ui': {
      'prompt_text': prompt_tabletop_ui,
      'model': MODEL_NAME,
      'max_tokens': 512,
      'temperature': 0,
      'query_prefix': '# ',
      'query_suffix': '.',
      'stop': ['#', 'objects = ['],
      'maintain_session': True,
      'debug_mode': False,
      'include_context': True,
      'has_return': False,
      'return_val_name': 'ret_val',
    },
    'parse_obj_name': {
      'prompt_text': prompt_parse_obj_name,
      'model': MODEL_NAME,
      'max_tokens': 512,
      'temperature': 0,
      'query_prefix': '# ',
      'query_suffix': '.',
      'stop': ['#', 'objects = ['],
      'maintain_session': False,
      'debug_mode': False,
      'include_context': True,
      'has_return': True,
      'return_val_name': 'ret_val',
    },
    'parse_position': {
      'prompt_text': prompt_parse_position,
      'model': MODEL_NAME,
      'max_tokens': 512,
      'temperature': 0,
      'query_prefix': '# ',
      'query_suffix': '.',
      'stop': ['#'],
      'maintain_session': False,
      'debug_mode': False,
      'include_context': True,
      'has_return': True,
      'return_val_name': 'ret_val',
    },
    'parse_question': {
      'prompt_text': prompt_parse_question,
      'model': MODEL_NAME,
      'max_tokens': 512,
      'temperature': 0,
      'query_prefix': '# ',
      'query_suffix': '.',
      'stop': ['#', 'objects = ['],
      'maintain_session': False,
      'debug_mode': False,
      'include_context': True,
      'has_return': True,
      'return_val_name': 'ret_val',
    },
    'transform_shape_pts': {
      'prompt_text': prompt_transform_shape_pts,
      'model': MODEL_NAME,
      'max_tokens': 512,
      'temperature': 0,
      'query_prefix': '# ',
      'query_suffix': '.',
      'stop': ['#'],
      'maintain_session': False,
      'debug_mode': False,
      'include_context': True,
      'has_return': True,
      'return_val_name': 'new_shape_pts',
    },
    'fgen': {
      'prompt_text': prompt_fgen,
      'model': MODEL_NAME,
      'max_tokens': 512,
      'temperature': 0,
      'query_prefix': '# define function: ',
      'query_suffix': '.',
      'stop': ['# define', '# example'],
      'maintain_session': False,
      'debug_mode': False,
      'include_context': True,
    }
  }
}

lmp_tabletop_coords = {
        'top_left':     (-0.3 + 0.05, -0.2 - 0.05),
        'top_side':     (0,           -0.2 - 0.05),
        'top_right':    (0.3 - 0.05,  -0.2 - 0.05),
        'left_side':    (-0.3 + 0.05, -0.5,      ),
        'middle':       (0,           -0.5,      ),
        'right_side':   (0.3 - 0.05,  -0.5,      ),
        'bottom_left':  (-0.3 + 0.05, -0.8 + 0.05),
        'bottom_side':  (0,           -0.8 + 0.05),
        'bottom_right': (0.3 - 0.05,  -0.8 + 0.05),
        'table_z':       0.0,
      }

## 10. すべての部品を組み立てる（`setup_LMP`）

ここで、これまで作った部品を結びつけます。

1. 環境をラップした `LMP_env`（ロボット API）を作る
2. **`fixed_vars`**：生成コードから常に使えるもの（`np` と shapely の全関数・クラス）
3. **`variable_vars`**：ロボット API（`get_obj_pos`, `put_first_on_second` など）と `say`（ロボットの発話。ここでは print するだけ）
4. 関数生成器 `lmp_fgen` を作る
5. 下位の LMP（`parse_obj_name`, `parse_position`, `parse_question`, `transform_shape_pts`）を作り、**`variable_vars` に関数として登録** する → 生成コードから呼び出せるようになる
6. 最上位の `tabletop_ui` LMP を作って返す

`variable_vars` は全 LMP で **同じ辞書オブジェクトを共有** しているので、ある LMP の実行中に生成された関数は、他の LMP からも再利用されます。

In [ ]:
def setup_LMP(env, cfg_tabletop):
  # LMP env wrapper
  cfg_tabletop = copy.deepcopy(cfg_tabletop)
  cfg_tabletop['env'] = dict()
  cfg_tabletop['env']['init_objs'] = list(env.obj_name_to_id.keys())
  cfg_tabletop['env']['coords'] = lmp_tabletop_coords
  LMP_env = LMP_wrapper(env, cfg_tabletop)
  # creating APIs that the LMPs can interact with
  fixed_vars = {
      'np': np
  }
  fixed_vars.update({
      name: eval(name)
      for name in shapely.geometry.__all__ + shapely.affinity.__all__
  })
  variable_vars = {
      k: getattr(LMP_env, k)
      for k in [
          'get_bbox', 'get_obj_pos', 'get_color', 'is_obj_visible', 'denormalize_xy',
          'put_first_on_second', 'get_obj_names',
          'get_corner_name', 'get_side_name',
      ]
  }
  variable_vars['say'] = lambda msg: print(f'robot says: {msg}')

  # creating the function-generating LMP
  lmp_fgen = LMPFGen(cfg_tabletop['lmps']['fgen'], fixed_vars, variable_vars)

  # creating other low-level LMPs
  variable_vars.update({
      k: LMP(k, cfg_tabletop['lmps'][k], lmp_fgen, fixed_vars, variable_vars)
      for k in ['parse_obj_name', 'parse_position', 'parse_question', 'transform_shape_pts']
  })

  # creating the LMP that deals w/ high-level language commands
  lmp_tabletop_ui = LMP(
      'tabletop_ui', cfg_tabletop['lmps']['tabletop_ui'], lmp_fgen, fixed_vars, variable_vars
  )

  return lmp_tabletop_ui

## 11. インタラクティブデモ

### 手順

1. 次のセルでブロックとボウルの数を決めて環境を初期化します。表示された **物体名（available objects）** を指示の中で使ってください。
2. その次のセルの `user_input` に指示を入力して実行すると、Gemini がロボットのコードを書き、シミュレーション動画が表示されます。

### 対応している指示の種類

- **空間的な推論**：「赤いブロックの左」「一番近い角」「一番遠いボウル」「右から 2 番目のブロック」
- **連続した動作**：「ブロックを同じ色のボウルに入れて」「右下の角にブロックを積んで」
- **文脈を使った指示**：「青いブロックでも同じことをして」「元に戻して」
- **言語的な推論**：「森の色のブロックを海の色のボウルに」
- **簡単な質問応答**：「青いボウルの左にブロックはいくつある？」

### 指示の例（物体名はシーンに合わせて変えてください）

- `put the sun-colored block on the bowl closest to it`
- `stack the blocks on the bottom most bowl`
- `arrange the blocks as a square in the middle`
- `move the square 5cm to the right`
- `how many blocks are to the right of the orange bowl?`
- `pick up the block closest to the top left corner and place it on the bottom right corner`

### 既知の制限

- 物体の位置はシミュレータの正確な値を使っているため、見た目が必要な指示（「一番暗いボウル」「一番大きい物体」など）には対応していません
- 物を置く動作は衝突判定をしていないので、物体が多いとぶつかることがあります
- 10 回以上続けて指示すると、実行履歴でプロンプトが長くなり、前半の例が無視されやすくなります（その場合は環境を初期化し直してください）
- 指示が曖昧だと意図どおりに動かないことがあります。言い換えてみてください（例：「the closest bowl」→「its closest bowl」）
- 生成されるコードはモデルや実行のたびに変わる可能性があります

### 11.1 環境の初期化

ブロックとボウルをランダムに選んで配置し、LMP を組み立て、カメラ画像を表示します。物体の配置を変えたいときや、実行履歴をリセットしたいときは、このセルを再実行してください。

In [ ]:
#@title Initialize Env { vertical-output: true }
num_blocks = 3 #@param {type:"slider", min:0, max:4, step:1}
num_bowls = 3 #@param {type:"slider", min:0, max:4, step:1}
high_resolution = False #@param {type:"boolean"}
high_frame_rate = False #@param {type:"boolean"}

# setup env and LMP
env = PickPlaceEnv(render=True, high_res=high_resolution, high_frame_rate=high_frame_rate)
block_list = np.random.choice(ALL_BLOCKS, size=num_blocks, replace=False).tolist()
bowl_list = np.random.choice(ALL_BOWLS, size=num_bowls, replace=False).tolist()
obj_list = block_list + bowl_list
_ = env.reset(obj_list)
lmp_tabletop_ui = setup_LMP(env, cfg_tabletop)

# display env
cv2_imshow(cv2.cvtColor(env.get_camera_image(), cv2.COLOR_BGR2RGB))

print('available objects:')
print(obj_list)

### 11.2 動画表示用の関数

シミュレーション中に保存した画像（`env.cache_video`）を mp4 にまとめ、Notebook 上で再生できるようにします（元の Notebook の `moviepy` の代わりです）。

In [ ]:
def show_video(frames, fps=25, path='demo.mp4', width=360):
    """画像のリストを mp4 に変換して Notebook 上に表示する。"""
    imageio.mimsave(path, frames, fps=fps)
    with open(path, 'rb') as f:
        b64 = base64.b64encode(f.read()).decode()
    display(HTML(
        f'<video width="{width}" controls autoplay loop muted>'
        f'<source src="data:video/mp4;base64,{b64}" type="video/mp4"></video>'
    ))

### 11.3 指示を入力して実行

`user_input` を書き換えて実行してください。出力には次の内容が順に表示されます。

- `LMP tabletop_ui exec:` … 最上位の LMP が生成したコード
- `LMP parse_position exec:` など … 下位の LMP が生成したコード
- `Creating function:` / `LMP FGEN created:` … 未定義だった関数を生成した様子
- `robot says:` … ロボットの発話
- 最後にシミュレーション動画

同じセルを続けて実行すると、それまでの指示の履歴を踏まえた指示（「元に戻して」など）ができます。

In [ ]:
#@title Interactive Demo { vertical-output: true }

user_input = 'Put the pink block to the left of the rightmost bowl' #@param {allow-input: true, type:"string"}

env.cache_video = []

print('Running policy and recording video...')
lmp_tabletop_ui(user_input, f'objects = {env.object_list}')

# 動画を表示
if env.cache_video:
  show_video(env.cache_video, fps=35 if high_frame_rate else 25)

### 11.4 （発展）生成されたコードを確認する

`tabletop_ui` の実行履歴（`exec_hist`）には、これまでに生成・実行したコードがすべて残っています。どのようなコードでロボットが動いたのかを振り返ってみましょう。

In [ ]:
print(highlight(lmp_tabletop_ui.exec_hist, PythonLexer(), TerminalFormatter()))

## まとめ

- Code as Policies は、**few-shot プロンプト + LLM のコード生成 + 実行** によって、自然言語の指示をロボットの動作に変換する
- LMP 同士を関数として呼び出せるようにすることで、「物体の特定」「座標の計算」などの部分問題を専門の LMP に任せられる
- 未定義の関数を LLM に書かせる **階層的コード生成** により、あらかじめ全ての関数を用意しなくてもよい
- 補完型モデル（davinci）からチャット型モデル（Gemini）に置き換える際は、**システム指示と出力の後処理** で「続きのコードだけ」を得る工夫が必要

**試してみると良いこと**

- `MODEL_NAME` を別の Gemini モデルに変えて、生成されるコードの違いを比べる
- `cfg_tabletop['lmps']['tabletop_ui']['debug_mode'] = True` にして、実行せずに生成コードだけ確認する
- `prompt_tabletop_ui` に新しい例を追加して、新しい種類の指示に対応させる